# 🚀 NLP Assignment 5: Fine-Tuning BERT for POS Tagging

---

## 🎯 Objective
Fine-tune a transformer model (DistilBERT) for:
- Part-of-Speech (POS) Tagging

---

## 🧠 Model Used
- DistilBERT (Hugging Face Transformers)

---

## 📊 Dataset Used
- Universal Dependencies (POS Tagging)

##Install libraries

In [1]:
!pip install datasets==2.16.1 transformers seqeval evaluate -q

##Import libraries

In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate

##Load Dataset

In [14]:
dataset = load_dataset("conll2003")
dataset

## Dataset Information

Dataset: CoNLL-2003

### Labels:
- POS Tags → pos
- Chunk Tags → chunk_tags
- Named Entity Tags → ner_tags

This dataset supports both POS tagging and Chunking.

In [14]:
label_list = dataset["train"].features["pos_tags"].feature.names

label_list

In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [14]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

##Load model

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs"
)

In [14]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    predictions = np.argmax(outputs.detach().numpy(), axis=2)

    predicted_labels = [label_list[p] for p in predictions[0]]

    for word, label in zip(tokens, predicted_labels):
        print(f"{word}: {label}")

In [ ]:
predict("John works at Google in California")